# Language Modeling

언어 모델은 앞에 나온 단어들을 보고 다음에 올 단어를 예측하는 모델이다. 예를 들어 `오늘 날씨가`라는 문맥이 주어졌을 때, 다음 단어로 `좋다`, `춥다`, `맑다` 같은 단어가 올 가능성을 계산한다.

이처럼 다음 단어를 예측할 수 있으면 문장 자동완성, 검색어 추천, 챗봇 응답 생성, 번역, 요약 같은 다양한 자연어 처리 작업으로 확장할 수 있다.

## 이미 학습된 언어 모델 체험

직접 언어 모델을 처음부터 학습하려면 많은 데이터와 시간이 필요하다. 그래서 여기서는 Hugging Face의 `transformers` 라이브러리에서 제공하는 GPT-2 모델을 불러와서 사용한다.

GPT-2는 앞 문맥을 보고 다음 토큰을 반복적으로 예측하여 문장을 생성하는 언어 모델이다.

> GPT-2 내부 구조는 Transformer 기반이지만, 이 파일에서는 구조를 깊게 다루지 않는다. 이후 Transformer 챕터에서 다시 연결해서 학습한다.

In [ ]:
# 필요 시 설치 
# %pip install transformers

In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# 사용할 사전 학습 모델 이름
model_name = 'gpt2'

# 생성 모델에서 패딩이 필요한 경우 왼쪽에 패딩을 붙이도록 설정하여 토크나이저 생성
tokenizer = GPT2Tokenizer.from_pretrained(model_name, padding_side='left')

# 다음 토큰을 예측하는 사전 학습 언어 모델
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT2는 기본 pad_token이 없으므로 eos_token을 pad_token처럼 사용하도록 지정
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## 입력 문장을 토큰으로 변환

In [3]:
# 입력 텍스트
input_text = "The future of AI is"

# tokenizer는 텍스트를 모델이 처리할 수 있는 숫자 형태로 변환
inputs = tokenizer(
    input_text,
    return_tensors='pt',
    return_attention_mask=True
)

print(inputs)

{'input_ids': tensor([[ 464, 2003,  286, 9552,  318]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}


In [5]:
# 정수 id 를 다시 토큰 문자열로 변환해서 확인
ids = inputs['input_ids'][0]
tokens = tokenizer.convert_ids_to_tokens(ids)

print('토큰 : ', tokens)

토큰 :  ['The', 'Ġfuture', 'Ġof', 'ĠAI', 'Ġis']


## 텍스트 생성

In [7]:
output = model.generate(
    input_ids=inputs['input_ids'],
    attention_mask=inputs['attention_mask'],
    pad_token_id=tokenizer.eos_token_id,
    max_length=50,
    num_return_sequences=1,          # 생성할 문장의 갯수
    do_sample=True,                  # 샘플링 여부, False로 설정할 경우 가장 확률이 높은 토큰을 선택하여 결과가 더 고정적
    temperature=0.8,                 # 생성의 무작위성 조절, 값이 낮으면 안정적이고 반복적 문장, 값이 높으면 다양하지만 어색한 문장
    top_k=50,                        # 다음 토큰 후보 상위 50개 제한
    top_p=0.95                       # 확률이 낮은 후보는 제외하여 생성 품질 안정화
)

print(output)

tensor([[  464,  2003,   286,  9552,   318,   287,   262,  1642,    13,  1081,
           351,   477,  3006,   286,  1597,    11,   340,   481,   307,  2408,
           284,  4331,   644,   318,  1016,   284,  1645,   351,   262,  1306,
          9552,  1910,    13,  2102,    11,  1813,   262, 32483,   286, 12779,
           326,  9552,  4394,    11,   340,   318,   407,  1598,   543,   481]])


In [8]:
# 생성 된 정수 토큰을 사람이 읽을 수 있는 문자열로 변환
result = tokenizer.decode(output[0], skip_special_tokens=True)  # 특수 토큰 출력에서 제거
print(result)

The future of AI is in the making. As with all areas of business, it will be difficult to predict what is going to happen with the next AI market. However, given the breadth of possibilities that AI offers, it is not clear which will


## 여러 문장 생성해보기

In [10]:
input_text = "Natural language processing helps computers"
inputs = tokenizer(input_text, return_tensors='pt', return_attention_mask=True)

outputs = model.generate(
    input_ids=inputs['input_ids'],
    attention_mask=inputs['attention_mask'],
    pad_token_id=tokenizer.eos_token_id,
    max_length=50,
    num_return_sequences=3,          # 생성할 문장의 갯수
    do_sample=True,                  # 샘플링 여부, False로 설정할 경우 가장 확률이 높은 토큰을 선택하여 결과가 더 고정적
    temperature=0.5,                 # 생성의 무작위성 조절, 값이 낮으면 안정적이고 반복적 문장, 값이 높으면 다양하지만 어색한 문장
    top_k=50,                        # 다음 토큰 후보 상위 50개 제한
    top_p=0.95                       # 확률이 낮은 후보는 제외하여 생성 품질 안정화
)

for i, output in enumerate(outputs, start=1):
    print(f'[{i}]')
    print(tokenizer.decode(output, skip_special_tokens=True))
    print()

[1]
Natural language processing helps computers learn to recognize and respond to complex language. In this paper, we present a new approach to learning to recognize and respond to complex language using a novel, natural language processing system. This system is based on the idea that the

[2]
Natural language processing helps computers understand how words are written.

The team also found that the team had been able to learn the grammar of more than 500 words from a dictionary. The language processing has been used in several languages, including English, German

[3]
Natural language processing helps computers understand and interpret human speech.

The researchers found that people who were familiar with the language learned to understand and understand language more quickly than people who were unfamiliar with the language.

"It's not just that people

